# Segmentación de clientes con KMeans
Proyecto de análisis no supervisado sobre el dataset Wholesale customers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme()


## 1. Carga del dataset


In [ ]:
data = pd.read_csv('/workspace/dataset.csv')
data.head()


In [ ]:
data.info()


In [ ]:
data.isnull().sum()


In [ ]:
data.describe()


## 2. Preprocesamiento
Se eliminan duplicados si existen y se escalan las variables con StandardScaler.


In [ ]:
data = data.drop_duplicates().reset_index(drop=True)
data.shape


In [ ]:
numeric_cols = data.columns.tolist()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data[numeric_cols])

data_scaled_df = pd.DataFrame(data_scaled, columns=numeric_cols)
data_scaled_df.head()


## 3. Método del codo


In [ ]:
inertias = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_scaled_df)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Método del codo')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.show()


## 4. Aplicación de KMeans
Se fija k=4 tras inspeccionar el codo.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(data_scaled_df)

data_clustered = data.copy()
data_clustered['Cluster'] = clusters

data_clustered.head()


In [ ]:
data_clustered['Cluster'].value_counts().sort_index()


## 5. Visualización con PCA


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(data_scaled_df)

pca_df = pd.DataFrame(pca_components, columns=['PC1', 'PC2'])
pca_df['Cluster'] = clusters

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='tab10')
plt.title('Clusters visualizados con PCA')
plt.show()


In [ ]:
explained_variance = pca.explained_variance_ratio_
explained_variance


## 6. Boxplots por clúster


In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=data_clustered, x='Cluster', y=col)
    plt.title(f'Boxplot de {col} por cluster')
    plt.show()


## 7. Promedio de cada característica por clúster


In [ ]:
cluster_means = data_clustered.groupby('Cluster')[numeric_cols].mean()
cluster_means


In [ ]:
cluster_means.T.plot(kind='bar', figsize=(12, 6))
plt.title('Promedio de variables por cluster')
plt.xlabel('Características')
plt.ylabel('Valor promedio')
plt.xticks(rotation=45)
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


## 8. Conclusión breve


In [ ]:
print('Se cargó el dataset con ruta absoluta, se comprobó la ausencia de nulos, se escalaron las variables con StandardScaler,')
print('se aplicó KMeans y se eligió k mediante el método del codo. Finalmente se visualizaron los clusters con PCA y')
print('se exploraron sus patrones con boxplots y promedios por cluster.')
